### Ingest Circuits File

Read circuits.csv, add ingestion metadata, and write the data to a Bronze Delta table.

In [0]:
%run ../00-common/01.environment-config


In [0]:
%run ../00-common/02.bronze-helpers

In [0]:
# SOURCE AND TARGET PATHS

source_file = f"{landing_folder_path}/circuits.csv"
target_path = f"{s3_root_path}/Bronze/Aaisha/circuits"

print("Source File :", source_file)
print("Target Path :", target_path)

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

circuits_schema = StructType([
    StructField("circuitId", StringType()),
    StructField("url", StringType()),
    StructField("circuitName", StringType()),
    StructField("lat", DoubleType()),
    StructField("long", DoubleType()),
    StructField("locality", StringType()),
    StructField("country", StringType())
])

In [0]:
circuits_df = (
    spark.read
        .format("csv")
        .option("header", "true")
        .option("mode", "FAILFAST")
        .schema(circuits_schema)
        .load(source_file)
)

display(circuits_df)

In [0]:
circuits_final_df = add_ingestion_metadata(circuits_df)

In [0]:
(
    circuits_final_df
        .write
        .format("delta")
        .mode("overwrite")
        .save(target_path)
)